## Multiple Tools for Agentic AI using Langraph

In [ ]:
from langchain_community.tools import ArxivQueryRun, WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper, ArxivAPIWrapper

In [ ]:
api_wrapper_arxiv = ArxivAPIWrapper(top_k_results=3, doc_content_chars_max=500)
arxiv= ArxivQueryRun(api_wrapper=api_wrapper_arxiv, description="use this tool to get the latest research papers from arxiv")
print(arxiv.name)
print(f"tool created: {arxiv.name}")
arxiv.invoke("Latest research papers on agriculture")

# basically we created an arxiv wrapper or tool which can be used later by agent.

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()

os.environ['TAVILY_API_KEY'] = os.getenv("TAVILY_API_KEY")
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")


In [ ]:
# Tavily Search Tool 
# It is used for web search  it can be later used by the agent 

from langchain_community.tools.tavily_search import TavilySearchResults

tavily = TavilySearchResults(description="use this tool to get the latest news and information from the web")
tavily.invoke("latest news on AI")

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.9)
llm.invoke("What is AI")
tools = [ tavily]

In [ ]:
llm_with_tools = llm.bind_tools(tools=tools) # bind_tools is used to bind the tools to the llm so that llm can use the tools.
llm_with_tools.invoke("What is AI and what are the latest news on AI")

## Langraph Workflow

In [ ]:
from typing_extensions import TypedDict # typedict means that we can define a dictionary with specific keys and value types. It allows us to create a structured data type that can be used for better type checking and code clarity.
from langchain_core.messages import AnyMessage 
from typing import Annotated # labelling a type with additional information.
from langgraph.graph.message import add_messages # Reducers in langGraph are used to process and transform messages as they flow through the graph.
# basically they are used for storing the messages in graph by appending them in dictironary format.

In [ ]:
class State(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages] # List fo all the messages in the conversation.
    # The add_messages reducer will be applied to this list of messages.
    #This creates a list to hold the conversation history. 
    # The add_messages reducer is highly important—it tells LangGraph to append new messages to the list rather than overwriting the old ones, keeping the full conversation history intact.
    

In [ ]:
from IPython.display import Image, display
from langgraph.prebuilt import ToolNode
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import tools_condition # pre made condition to check if the tool is present in the messages or not.


def tool_calling_llm(state: State):
    return {"messages": [llm_with_tools.invoke(state["messages"])]} # This passes the entire message history to your GPT-3.5 model (which has the tools bound to it). The LLM processes it and generates a response. 
# That response is then returned in a dictionary, which LangGraph uses to update the global State.


builder = StateGraph(State)
builder.add_node("tool_calling_llm", tool_calling_llm)
builder.add_node("tools", ToolNode(tools)) # ToolNode is a special type of node in LangGraph that is designed to execute tools. By adding a ToolNode with the list of tools, we are telling LangGraph that when it reaches this node, it should check the conditions and execute the appropriate tool if the conditions are met.


builder.add_edge(START, "tool_calling_llm")
builder.add_conditional_edges("tool_calling_llm", tools_condition) # tools_condition is premade condition that checks if any of the tools are mentioned in the messages. If the condition is satisfied, it will direct the flow to the "tools" node where the tools will be executed. If not, it will continue with the normal flow of the conversation.
builder.add_edge("tools", "tool_calling_llm") # matlab "tools" thi lai ne pachi "tool_calling_llm" node par javanu che, jethi ke conversation flow chalu rahe.
builder.add_edge("tool_calling_llm", END)

graph_builder = builder.compile()
display(Image(graph_builder.get_graph().draw_mermaid_png()))

# basicllay in this cell we are creating a graph where we have a node called "tool_calling_llm" which will call the llm with tools and then we have another node called "tools" which will execute the tools if the condition is satisfied.
#  The condition is checked using the tools_condition which checks if the tool is present in the messages or not. Finally we have edges connecting the nodes and we compile the graph.

In [ ]:
messages = graph_builder.invoke({"messages":"1706.03762"})
for m in messages["messages"]:
    m.pretty_print()


## Memory in LangGraph

In [ ]:
from IPython.display import Image, display
from langgraph.prebuilt import ToolNode
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import tools_condition # pre made condition to check if the tool is present in the messages or not.
from langgraph.checkpoint.memory import InMemorySaver # InMemorySaver is a simple implementation of a memory saver that stores the conversation history in memory. It allows us to keep track of the conversation history and use it for future interactions.

memory = InMemorySaver() # passed it downwards in graphbuilder .

def tool_calling_llm(state: State):
    return {"messages": [llm_with_tools.invoke(state["messages"])]} # This passes the entire message history to your GPT-3.5 model (which has the tools bound to it). The LLM processes it and generates a response. 
# That response is then returned in a dictionary, which LangGraph uses to update the global State.

builder = StateGraph(State)
builder.add_node("tool_calling_llm", tool_calling_llm)
builder.add_node("tools", ToolNode(tools)) # ToolNode is a special type of node in LangGraph that is designed to execute tools. By adding a ToolNode with the list of tools, we are telling LangGraph that when it reaches this node, it should check the conditions and execute the appropriate tool if the conditions are met.

builder.add_edge(START, "tool_calling_llm")
builder.add_conditional_edges("tool_calling_llm", tools_condition) # tools_condition is premade condition that checks if any of the tools are mentioned in the messages. If the condition is satisfied, it will direct the flow to the "tools" node where the tools will be executed. If not, it will continue with the normal flow of the conversation.
builder.add_edge("tools", "tool_calling_llm") # matlab "tools" thi lai ne pachi "tool_calling_llm" node par javanu che, jethi ke conversation flow chalu rahe.
builder.add_edge("tool_calling_llm", END)

graph_builder = builder.compile(checkpointer=memory)# here we are passing the memory saver to the compile function, which allows us to save the conversation history in memory. This way, we can keep track of the conversation history and use it for future interactions.
display(Image(graph_builder.get_graph().draw_mermaid_png()))

# basicllay in this cell we are creating a graph where we have a node called "tool_calling_llm" which will call the llm with tools and then we have another node called "tools" which will execute the tools if the condition is satisfied.
#  The condition is checked using the tools_condition which checks if the tool is present in the messages or not. Finally we have edges connecting the nodes and we compile the graph.

In [ ]:
config = {"configurable":{"thread_id": "1"}} # 
response = graph_builder.invoke({"messages":"Hello my name is jenil "}, config=config)
response

In [ ]:
response['messages'][-1]

In [ ]:
response = graph_builder.invoke({"messages":"what is my name ?"}, config=config)
print(response["messages"][-1])

## Streaming in LangGraph

Methods : .stream() and astream()
These methods area sync and async methods for streaming back results


In [ ]:
# Streaming in langgrpah means that we can get the response from the llm in chunks rather than waiting for the entire response to be generated.
#  This allows us to display the response to the user as it is being generated, which can improve the user experience.

from langgraph.checkpoint.memory import MemorySaver
memory = MemorySaver() # creating a memory saver instance to save the conversation history in memory.
# This allows us to keep track of the conversation history and use it for future interactions.

In [ ]:
def superbot(state:State):
    return {"messages":[llm.invoke(state["messages"])]}

In [ ]:
graph = StateGraph(State)
graph.add_node("superbot", superbot)
graph.add_edge(START, "superbot")
graph.add_edge("superbot", END)

graph_builder = graph.compile(checkpointer=memory)

from IPython.display import Image, display
display(Image(graph_builder.get_graph().draw_mermaid_png()))

In [ ]:
# invocation
config = {"configurable":{"thread_id": "1"}}
graph_builder.invoke({"messages":"Hello"}, config=config)

## Human in the Loop

In [ ]:
from langgraph.types import Command , interrupt
from langchain_core.tools import tool
class state(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    # The add_messages reducer will be applied to this list of messages.
    #This creates a list to hold the conversation history. 
    # The add_messages reducer is highly important—it tells LangGraph to append new messages to the list rather than overwriting the old ones, keeping the full conversation history intact.

    graph_builder = StateGraph(State)

    @tool # basically we created the tool(human_assistance) using the tool decorator which allows us to use this function as a tool in our graph.
    def human_assistance(query: str)-> str:
        """request assistance from human"""
        human_response = interrupt("query " + query) # interrupt is a special command in LangGraph that allows the agent to pause its execution and request input from a human. 
        # When the agent reaches this point, it will send a message to the human operator with the query, and wait for a response. Once the human provides an answer, the agent will resume its execution with that input.
        return human_response['data']
    
    tool = tavily(max_results=2)
    tools = [tool,human_assistance]
    llm_with_tools = llm.bind_tools(tools=tools)

    def chatbot(state:State):
        messsage = llm_with_tools.invoke(state['messages'])

        return {"messages":[messsage]}
    
    graph_builder.add_node("chatbot", chatbot)
    tool_node = ToolNode(tools=tools)
    graph_builder.add_node("tools", tool_node)

    graph_builder.add_conditional_edges("chatbot", tools_condition)
    graph_builder.add_edge("tools", "chatbot")
    graph_builder.add_edge(START, "chatbot")
